---

# 🔍 Esploriamo i Dati Preprocessati

Vediamo cosa contengono i file processati e come sono strutturati.

In [ ]:
# Setup: aggiungi src/ al path per importare i moduli
import sys
from pathlib import Path

# Aggiungi la directory src al path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

# Import librerie
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Import configurazione progetto
import config

# Configurazione plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Setup completato!")
print(f"📁 Project root: {project_root}")
print(f"📊 Periodi da analizzare: {config.PERIODS}")

## 1. Caricamento Dati Preprocessati

I file in `data/processed/` contengono **un token per riga** (già puliti e lemmatizzati).

In [ ]:
def load_tokens(period):
    """Carica i token preprocessati per un periodo."""
    file_path = config.get_period_file_path(period, 'processed')
    
    with open(file_path, 'r', encoding='utf-8') as f:
        tokens = [line.strip() for line in f if line.strip()]
    
    return tokens

# Carica tutti i periodi
data = {}
for period in config.PERIODS:
    data[period] = load_tokens(period)
    print(f"{period}: {len(data[period]):,} tokens")

print(f"\n✅ Caricati {sum(len(tokens) for tokens in data.values()):,} tokens totali")

## 2. Statistiche Descrittive

Analizziamo la distribuzione dei token e del vocabolario.

In [ ]:
# Crea DataFrame con statistiche
stats_list = []

for period, tokens in data.items():
    vocab = set(tokens)
    counter = Counter(tokens)
    
    stats_list.append({
        'Periodo': period,
        'Tokens Totali': len(tokens),
        'Vocabolario': len(vocab),
        'Lunghezza Media': np.mean([len(t) for t in tokens]),
        'Top Parola': counter.most_common(1)[0][0],
        'Freq Top Parola': counter.most_common(1)[0][1]
    })

df_stats = pd.DataFrame(stats_list)
df_stats

In [ ]:
# Visualizza crescita del vocabolario nel tempo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Numero di tokens per periodo
axes[0].bar(df_stats['Periodo'], df_stats['Tokens Totali'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Periodo', fontsize=12)
axes[0].set_ylabel('Numero di Tokens', fontsize=12)
axes[0].set_title('Distribuzione Tokens per Periodo', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Grafico 2: Crescita vocabolario nel tempo
axes[1].plot(df_stats['Periodo'], df_stats['Vocabolario'], 
             marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[1].set_xlabel('Periodo', fontsize=12)
axes[1].set_ylabel('Dimensione Vocabolario', fontsize=12)
axes[1].set_title('Crescita Vocabolario nel Tempo', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 Osservazione: Il vocabolario cresce leggermente nel tempo (nuove parole tecnologiche)")

## 3. Analisi Vocabolario Globale

Vediamo quante parole uniche abbiamo in totale e quali sono le più frequenti.

In [ ]:
# Unisci tutti i token
all_tokens = []
for tokens in data.values():
    all_tokens.extend(tokens)

# Vocabolario globale
global_vocab = set(all_tokens)
global_counter = Counter(all_tokens)

print(f"🌍 Vocabolario Globale: {len(global_vocab)} parole uniche")
print(f"📊 Tokens Totali: {len(all_tokens):,}")
print(f"\n🔝 Top 20 Parole Più Frequenti:\n")

top_20 = global_counter.most_common(20)
for i, (word, freq) in enumerate(top_20, 1):
    print(f"{i:2d}. {word:<15} → {freq:>8,} occorrenze")

In [ ]:
# Visualizza distribuzione frequenze (Zipf's Law)
frequencies = sorted(global_counter.values(), reverse=True)
ranks = range(1, len(frequencies) + 1)

plt.figure(figsize=(10, 6))
plt.loglog(ranks, frequencies, marker='o', linestyle='none', alpha=0.6)
plt.xlabel('Rank (log scale)', fontsize=12)
plt.ylabel('Frequenza (log scale)', fontsize=12)
plt.title("Distribuzione di Zipf - Legge Potenza delle Frequenze", fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📉 Zipf's Law: poche parole molto frequenti, molte parole rare")

## 4. Parole Interessanti per Semantic Drift

Identifichiamo parole che appaiono in tutti i periodi (candidati per analisi drift).

In [ ]:
# Per ogni parola, controlla in quanti periodi appare
word_periods = {}

for word in global_vocab:
    periods_with_word = []
    for period, tokens in data.items():
        if word in tokens:
            periods_with_word.append(period)
    
    word_periods[word] = periods_with_word

# Parole che appaiono in TUTTI i periodi
words_all_periods = [word for word, periods in word_periods.items() 
                     if len(periods) == len(config.PERIODS)]

print(f"📌 Parole presenti in TUTTI i {len(config.PERIODS)} periodi: {len(words_all_periods)}")
print(f"\nEsempi: {sorted(words_all_periods)[:20]}")

# Parole con semantic drift noto (se presenti)
drift_words = ['gay', 'cell', 'mouse', 'web', 'cloud', 'tweet', 'viral']
drift_words_present = [w for w in drift_words if w in global_vocab]

print(f"\n🎯 Parole con semantic drift conosciuto (presenti nel dataset):")
for word in drift_words_present:
    periods = word_periods.get(word, [])
    print(f"  - '{word}': presente in {len(periods)}/{len(config.PERIODS)} periodi → {periods}")

## 5. Anteprima Frequenze per Periodo

Vediamo come cambia la frequenza di alcune parole nel tempo.

In [ ]:
# Seleziona alcune parole interessanti
words_to_track = ['gay', 'cell', 'mouse', 'computer', 'internet', 'war', 'peace']
words_to_track = [w for w in words_to_track if w in global_vocab]  # Filtra solo presenti

# Calcola frequenze per periodo
freq_over_time = {word: [] for word in words_to_track}

for period in config.PERIODS:
    counter = Counter(data[period])
    total = len(data[period])
    
    for word in words_to_track:
        # Frequenza relativa (per 10k tokens)
        freq = (counter.get(word, 0) / total) * 10000
        freq_over_time[word].append(freq)

# Plot
plt.figure(figsize=(12, 6))

for word in words_to_track:
    plt.plot(config.PERIODS, freq_over_time[word], 
             marker='o', linewidth=2, label=word)

plt.xlabel('Periodo', fontsize=12)
plt.ylabel('Frequenza (per 10k tokens)', fontsize=12)
plt.title('Evoluzione Frequenza Parole nel Tempo', fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='best')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Nota: Le variazioni di frequenza possono indicare cambiamenti d'uso o significato")

---

## 📝 Riepilogo

### Cosa abbiamo visto:

1. ✅ **Dati preprocessati** pronti per il training
   - ~500k tokens per periodo
   - 113 parole uniche totali (dataset demo)

2. ✅ **Distribuzione uniforme** dei dati tra periodi

3. ✅ **Vocabolario in crescita** nel tempo (nuove parole tecniche)

4. ✅ **Parole candidate** per analisi semantic drift identificate

---

## 🚀 Prossimi Passi

### FASE 3: Costruzione Vocabolario
- Creare vocabolario globale unificato
- Mappare parole → indici numerici
- Gestire token rari (soglia frequenza)
- Salvare vocabolario serializzato

### FASE 4: Word Embeddings (PyTorch)
- Implementare modello CBOW/Skip-gram
- Training separato per ogni periodo
- Salvare embeddings allenati

### FASE 5+: Allineamento e Analisi
- Orthogonal Procrustes alignment
- Calcolo semantic drift
- Visualizzazioni t-SNE/PCA

---

## 🎓 Riferimenti e Note Tecniche

### Cos'è un Word Embedding?

Un **embedding** è una rappresentazione vettoriale di una parola:
- Invece di "cat" → usiamo un vettore di 300 numeri: `[0.23, -0.45, 0.12, ...]`
- Parole simili hanno vettori vicini nello spazio
- Permette operazioni matematiche: `king - man + woman ≈ queen`

### Perché allineare gli spazi?

Gli embeddings di periodi diversi sono in "spazi" diversi:
- `gay_1900s` potrebbe essere `[0.5, 0.3, ...]`
- `gay_1990s` potrebbe essere `[0.1, -0.8, ...]`

**Senza allineamento** non possiamo confrontarli!

**Con Orthogonal Procrustes** ruotiamo/trasliamo uno spazio sull'altro → confronto possibile.

### Metric: Cosine Distance

Semantic drift = quanto è cambiata la parola?

```
drift = 1 - cosine_similarity(vec_1900s, vec_1990s_aligned)
```

- `drift = 0` → nessun cambiamento
- `drift = 1` → significato completamente diverso